In [3]:
!pip install tqdm gensim nltk rapidfuzz scikit-learn pandas


   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 388.3 kB/s eta 0:01:02
    --------------------------------------- 0.5/24.4 MB 388.3 kB/s eta 0:01:02
    --------------------------------------- 0.5/24.4 MB 388.3 kB/s eta 0

In [5]:
import sys
!{sys.executable} -m pip install tqdm


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [7]:
import sys
!{sys.executable} -m pip install gensim


  Using cached smart_open-7.3.1-py3-none-any.whl.metadata (24 kB)
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 239.6 kB/s eta 0:01:40
    -----------

In [9]:
import sys
!{sys.executable} -m pip install tqdm gensim nltk rapidfuzz scikit-learn pandas


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------ ------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [19]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Laptopkaran\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Laptopkaran\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [20]:
import os
import re
import random
import warnings
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

import nltk
from gensim.models import Word2Vec
from rapidfuzz import fuzz

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

In [21]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

W2V_SAMPLE_SIZE = 200000
W2V_VECTOR_SIZE = 100
W2V_WINDOW = 5
W2V_MIN_COUNT = 5
W2V_TOPN = 30
LEV_THRESHOLD = 85        
MIN_SAMPLES_TO_TRAIN = 30     

train_path  = r"C:\Users\Laptopkaran\Downloads\train_data.csv"
test_path   = r"C:\Users\Laptopkaran\Downloads\test_data.csv"
titles_path = r"C:\Users\Laptopkaran\Downloads\title_brand.csv"

out_warranty_csv = r"C:\Users\Laptopkaran\Downloads\warranty_sentiment_result.csv"
out_submission_csv = r"C:\Users\Laptopkaran\Downloads\q2_submission.csv"

warnings.filterwarnings("ignore")


try:
    nltk.data.find('tokenizers/punkt')
except Exception:
    nltk.download('punkt')

def simple_tokenize(text):

    text = str(text).lower()
    text = re.sub(r'[^a-z0-9]', ' ', text)
    tokens = nltk.word_tokenize(text)
    return [t for t in tokens if len(t) > 1]


def safe_read_csv(path, **kwargs):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    return pd.read_csv(path, low_memory=False, **kwargs)

print("Loading files...")
train = safe_read_csv(train_path)
test  = safe_read_csv(test_path)
titles = safe_read_csv(titles_path)

print("Shapes:", {"train": train.shape, "test": test.shape, "titles": titles.shape})
print("Sample columns (train/test/titles):")
print(list(train.columns)[:50])
print(list(test.columns)[:50])
print(list(titles.columns)[:50])

if 'Asin' in titles.columns and 'asin' not in titles.columns:
    titles = titles.rename(columns={'Asin':'asin'})

if 'reviewText' not in train.columns or 'overall' not in train.columns:
    raise KeyError("train باید ستون‌های 'reviewText' و 'overall' را داشته باشد.")
if 'reviewText' not in test.columns:
    raise KeyError("test باید ستون 'reviewText' را داشته باشد.")

# حذف ردیف‌هایی که متن یا امتیاز ندارند
train = train.dropna(subset=['reviewText', 'overall']).copy()
test  = test.dropna(subset=['reviewText']).copy()


train['overall_raw'] = train['overall'].astype(float)
train['overall'] = ((train['overall_raw'] - 1.0) / 4.0) * 5.0
train['overall'] = train['overall'].clip(lower=0.0, upper=5.0)


base_words = ['warranty', 'guarantee']
common_typos = [
    'warrenty','waranty','warantee','warrantee','warenty',
    'guarentee','guaranty','guaranttee','guarenty',
    'garranty','wartanty'
]
similar_words = set(base_words + common_typos)

print("Preparing sample for Word2Vec...")
sample_size = min(W2V_SAMPLE_SIZE, len(train))
train_sample = train['reviewText'].sample(sample_size, random_state=SEED).astype(str).tolist()

print("Tokenizing sample for Word2Vec...")
sentences = [simple_tokenize(txt) for txt in tqdm(train_sample, desc="Tokenizing")]

w2v_model = None
if len(sentences) > 10:
    print(f"Training Word2Vec on {len(sentences)} sentences ...")
    w2v_model = Word2Vec(sentences,
                         vector_size=W2V_VECTOR_SIZE,
                         window=W2V_WINDOW,
                         min_count=W2V_MIN_COUNT,
                         workers=4,
                         seed=SEED)
    for bw in base_words:
        try:
            if bw in w2v_model.wv.key_to_index:
                most_sim = w2v_model.wv.most_similar(bw, topn=W2V_TOPN)
                for w, score in most_sim:
                    if re.match(r'^[a-z]{2,}$', w):
                        similar_words.add(w)
        except KeyError:
            pass
    vocab = list(w2v_model.wv.key_to_index.keys())
    for v in vocab:
        for base in base_words:
            if fuzz.ratio(v, base) >= LEV_THRESHOLD:
                similar_words.add(v)
else:
    print("Not enough data to train Word2Vec; skipping W2V step.")

keywords = sorted(similar_words)
print(f"Found {len(keywords)} warranty-related keywords (sample): {keywords[:40]}")

MAX_KEYWORDS_FOR_REGEX = 1500
use_keywords = keywords[:MAX_KEYWORDS_FOR_REGEX]
pattern = r'\b(?:' + '|'.join(map(re.escape, use_keywords)) + r')\b'
pattern_re = re.compile(pattern, flags=re.IGNORECASE)


print("Marking mentions of warranty in train and test...")
train['mentions_warranty'] = train['reviewText'].astype(str).progress_apply(lambda x: bool(pattern_re.search(x)))
test['mentions_warranty']  = test['reviewText'].astype(str).progress_apply(lambda x: bool(pattern_re.search(x)))

def fuzzy_token_match(text, base_list=base_words, threshold=LEV_THRESHOLD):
    tokens = simple_tokenize(text)
    for t in tokens:
        for b in base_list:
            if fuzz.ratio(t, b) >= threshold:
                return True
    return False

train['mentions_warranty'] = train.apply(
    lambda r: r['mentions_warranty'] or fuzzy_token_match(r['reviewText']), axis=1
)
test['mentions_warranty'] = test.apply(
    lambda r: r['mentions_warranty'] or fuzzy_token_match(r['reviewText']), axis=1
)

train_warranty = train[train['mentions_warranty']].copy()
test_warranty  = test[test['mentions_warranty']].copy()

print("Counts -> train warranty:", len(train_warranty), " test warranty:", len(test_warranty))


if 'asin' in train_warranty.columns:
    mean_by_asin = train_warranty.groupby('asin')['overall'].mean().reset_index().rename(columns={'overall':'mean_warranty_rating'})
else:
    mean_by_asin = pd.DataFrame(columns=['asin','mean_warranty_rating'])

if 'asin' in titles.columns:
    final_analytics = mean_by_asin.merge(titles, on='asin', how='left')
else:
    final_analytics = mean_by_asin.copy()

final_analytics.to_csv(out_warranty_csv, index=False, encoding='utf-8-sig')
print("Saved:", out_warranty_csv)


classifier = None
if len(train_warranty) < MIN_SAMPLES_TO_TRAIN:
    print(f"Too few warranty reviews ({len(train_warranty)}) to train a reliable classifier. Skipping training.")
else:
    train_warranty['label'] = train_warranty['overall'].round().astype(int).clip(0,5)

    X = train_warranty['reviewText'].astype(str).values
    y = train_warranty['label'].values

    try:
        X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=y)
    except Exception:
        X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.15, random_state=SEED)

    vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
    X_tr_vec = vectorizer.fit_transform(X_tr)
    X_val_vec = vectorizer.transform(X_val)

    clf = LogisticRegression(max_iter=2000, multi_class='multinomial', random_state=SEED)
    clf.fit(X_tr_vec, y_tr)

    y_val_pred = clf.predict(X_val_vec)
    print("Validation classification report (on warranty subset):")
    print(classification_report(y_val, y_val_pred, digits=4))
    print("Validation micro-F1:", f1_score(y_val, y_val_pred, average='micro'))

    classifier = (vectorizer, clf)


global_mode = 3
if 'overall' in train.columns and not train['overall'].isna().all():
    try:
        gm = float(train['overall'].mode().iloc[0])
        global_mode = int(round(gm))
    except Exception:
        global_mode = 3
global_mode = max(0, min(5, global_mode))

asin_mean_map = {}
if 'asin' in mean_by_asin.columns:
    asin_mean_map = mean_by_asin.set_index('asin')['mean_warranty_rating'].to_dict()

def predict_row(row):
    if row.get('mentions_warranty', False) and classifier is not None:
        vec, model_clf = classifier
        x = vec.transform([str(row['reviewText'])])
        try:
            p = int(model_clf.predict(x)[0])
            return max(0, min(5, p))
        except Exception:
            pass
    asin = row.get('asin', None)
    if pd.notna(asin) and asin in asin_mean_map:
        return int(round(asin_mean_map[asin]))
    # fallback -> global_mode
    return int(global_mode)


print("Predicting on test set...")
if 'predicted' in test.columns:
    test = test.drop(columns=['predicted'])

test['predicted'] = test.progress_apply(predict_row, axis=1)

submission = test[['predicted']].copy()
submission['predicted'] = submission['predicted'].astype(int).clip(0,5)

submission.to_csv(out_submission_csv, index=False, encoding='utf-8-sig')
print("Saved submission:", out_submission_csv)
print("Predicted value counts:\n", submission['predicted'].value_counts().sort_index())

print("train shape:", train.shape)
print("train_warranty shape:", train_warranty.shape)
print("test shape:", test.shape)
print("num unique asin with warranty mean:", len(mean_by_asin))
print("global_mode used as fallback:", global_mode)
print("Files produced:")
print(" -", out_warranty_csv)
print(" -", out_submission_csv)


Loading files...
Shapes: {'train': (838944, 11), 'test': (20000, 10), 'titles': (786445, 3)}
Sample columns (train/test/titles):
['overall', 'vote', 'verified', 'reviewTime', 'reviewerID', 'asin', 'style', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime']
['vote', 'verified', 'reviewTime', 'reviewerID', 'asin', 'style', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime']
['asin', 'title', 'brand']
Preparing sample for Word2Vec...
Tokenizing sample for Word2Vec...



Tokenizing: 100%|████████████████████████████████████████████████████████████| 200000/200000 [00:51<00:00, 3906.52it/s]


Training Word2Vec on 200000 sentences ...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Found 60 warranty-related keywords (sample): ['affiliation', 'assure', 'asurion', 'blame', 'claim', 'commitment', 'company', 'contract', 'dealer', 'deserve', 'disappoint', 'dissapoint', 'doubt', 'expire', 'expired', 'garranty', 'guarantee', 'guaranteed', 'guarantees', 'guaranttee', 'guaranty', 'guarentee', 'guarenty', 'honor', 'honored', 'influence', 'insurance', 'lifetime', 'manufacturer', 'offer', 'policy', 'promise', 'promises', 'rebate', 'receipt', 'refund', 'reimbursement', 'repairs', 'reply', 'request']
Marking mentions of warranty in train and test...



%|                                                                                       | 0/838944 [00:00<?, ?it/s]
%|                                                                           | 993/838944 [00:00<01:24, 9929.65it/s]
%|▏                                                                        | 2111/838944 [00:00<01:18, 10637.56it/s]
%|▎                                                                         | 3175/838944 [00:00<01:23, 9957.43it/s]
%|▎                                                                         | 4176/838944 [00:00<01:24, 9902.28it/s]
%|▍                                                                        | 5312/838944 [00:00<01:20, 10375.37it/s]
%|▌                                                                        | 6353/838944 [00:00<01:20, 10318.34it/s]
%|▋                                                                         | 7387/838944 [00:00<01:23, 9912.43it/s]
%|▋                                                            

Counts -> train warranty: 120909  test warranty: 3869
Saved: C:\Users\Laptopkaran\Downloads\warranty_sentiment_result.csv
Validation classification report (on warranty subset):
              precision    recall  f1-score   support

           0     0.6611    0.8171    0.7309      3975
           1     0.4116    0.1991    0.2684      1919
           2     0.3929    0.2320    0.2918      1991
           4     0.4544    0.3370    0.3870      2884
           5     0.7278    0.8871    0.7996      7368

    accuracy                         0.6396     18137
   macro avg     0.5296    0.4945    0.4955     18137
weighted avg     0.5995    0.6396    0.6070     18137

Validation micro-F1: 0.6395765562110602
Predicting on test set...



%|                                                                                        | 0/20000 [00:00<?, ?it/s]
%|█▋                                                                          | 446/20000 [00:00<00:04, 4417.64it/s]
%|███▍                                                                        | 913/20000 [00:00<00:04, 4538.90it/s]
%|█████▏                                                                     | 1372/20000 [00:00<00:04, 4541.49it/s]
%|██████▉                                                                    | 1836/20000 [00:00<00:03, 4580.14it/s]
%|████████▌                                                                  | 2295/20000 [00:00<00:03, 4567.28it/s]
%|██████████▎                                                                | 2752/20000 [00:00<00:03, 4568.20it/s]
%|████████████                                                               | 3209/20000 [00:00<00:03, 4496.98it/s]
%|█████████████▊                                               

Saved submission: C:\Users\Laptopkaran\Downloads\q2_submission.csv
Predicted value counts:
 predicted
0    2334
1    1261
2    3513
3    3395
4    3427
5    6070
Name: count, dtype: int64
train shape: (838944, 13)
train_warranty shape: (120909, 14)
test shape: (20000, 12)
num unique asin with warranty mean: 37905
global_mode used as fallback: 5
Files produced:
 - C:\Users\Laptopkaran\Downloads\warranty_sentiment_result.csv
 - C:\Users\Laptopkaran\Downloads\q2_submission.csv
